# PR Generator (Cleaned)
Because of how messy and disorganized the previous code was, it made it really difficult to follow along to what we are actually trying to achieve. This code will have full annotations and more organized efforts to explain what is actually happening at each step. 

## 1: Spreadsheet Assembly
- Utilizes Emilio + Jake code to create proper spreadsheets of tournaments and players. 
- Annotated each way 

### 1.1: Informational Setup
Here, the date and GraphQL query will be set up and generated, and all variable definitions will take place. 

In [ ]:
# libraries
import requests
import json
import pandas as pd
from keys import api_key, huggingface_key, openai_key
import os
from datetime import datetime
from collections import Counter
from queries import *
from processing import *
import ast
from openai import OpenAI, completions
from collections import defaultdict

In [ ]:
# variables and keys for tourney scraping
AUTH_TOKEN = api_key
SF_BASED_COORDS = "37.77151615492457, -122.41563048985462"
SF_RADIUS = "70mi"

SAC_BASED_COORDS = "38.57608096237729, -121.49183616631059"
SAC_RADIUS = "40mi"

NUM_PER_PAGE = 50
request_url = 'https://api.start.gg/gql/alpha'

In [ ]:
# graphQL query generation
def generate_graphql_query(timestamps):
    template = """
query BayNorCalTournaments($page: Int, $perPage: Int, $coordinates: String!, $radius: String!) {{
  tournaments(
    query: {{
    page: $page
    perPage: $perPage
    filter: {{
      location: {{
        distanceFrom: $coordinates,
        distance: $radius
      }},
      afterDate: {after_date} 
      beforeDate: {before_date}
    }}
    sortBy:"startAt"
  }}) {{
    nodes {{
      id
      name
      city
      slug
      startAt
      events {{
        slug
        numEntrants
        videogame {{
          name
        }}
      }}
    }}
  }}
}}
"""
    after_date = timestamps[0]
    before_date = timestamps[1]
    query = template.format(after_date=after_date, before_date=before_date)

    return query

In [ ]:
# NL date to unix
def date_to_unix(date_string, date_format="%Y-%m-%d"):
    try:
        date_obj = datetime.strptime(date_string, date_format)
        unix_timestamp = int(date_obj.timestamp())
        return unix_timestamp
    except ValueError as e:
        print(f"Error: {e}")
        return None

In [ ]:
# define date ranges
start_date = "2025-01-01"
end_date = "2025-03-31"
timestamps = [date_to_unix(start_date), date_to_unix(end_date)]
# timestamps

In [ ]:
# generate graphQL query
query = generate_graphql_query(timestamps)
# print(query)

### 1.2: Spreadsheet Generation

Here, the functions needed to generate the spreadsheet are defined, and the spreadsheet/df is then generated.

In [ ]:
# tourney retrieval function - utilizes query to return df
def get_all_tournies(auth_token, query, coords, radius, num_per_page):
  
  graphql_query = query 
  tournies = []

  for i in range(1, 10):
    variables = {
        "page": i,
        "perPage": num_per_page,
        "coordinates": coords,
        "radius": "50mi"
    }
    data = {"query" : graphql_query, "variables": variables}
    json_data = json.dumps(data)
    auth_header = auth_token
    header = {'Authorization': 'Bearer ' + auth_header}  

    response = requests.post(url=request_url, headers=header, data=json_data)
    json_resp = json.loads(response.text)
    print(json_resp)
    if ("errors" not in json_resp):
       
      curr_tournies_page = json_resp['data']['tournaments']['nodes']
      print("Number of tournies in page is:" + str(len(curr_tournies_page)))

      tournies += curr_tournies_page
  return tournies

In [ ]:
# flatten df after tourneys are retrieved 
def flatten_nested_json_df(df):
    
    df = df.reset_index()
    
    print(f"original shape: {df.shape}")
    print(f"original columns: {df.columns}")
    
    s = (df.applymap(type) == list).all()
    list_columns = s[s].index.tolist()
    
    s = (df.applymap(type) == dict).all()
    dict_columns = s[s].index.tolist()
    
    print(f"lists: {list_columns}, dicts: {dict_columns}")
    while len(list_columns) > 0 or len(dict_columns) > 0:
        new_columns = []
        
        for col in dict_columns:
            print(f"flattening: {col}")
            horiz_exploded = pd.json_normalize(df[col]).add_prefix(f'{col}.')
            horiz_exploded.index = df.index
            df = pd.concat([df, horiz_exploded], axis=1).drop(columns=[col])
            new_columns.extend(horiz_exploded.columns) 
        
        for col in list_columns:
            print(f"exploding: {col}")
            df = df.drop(columns=[col]).join(df[col].explode().to_frame())
            df = df.reset_index(drop=True)
            new_columns.append(col)
        
        s = (df[new_columns].applymap(type) == list).all()
        list_columns = s[s].index.tolist()

        s = (df[new_columns].applymap(type) == dict).all()
        dict_columns = s[s].index.tolist()
        
        print(f"lists: {list_columns}, dicts: {dict_columns}")
        
    print(f"final shape: {df.shape}")
    print(f"final columns: {df.columns}")
    return df

In [ ]:
# generate the spreadsheet/scrape data
tournies = []
bay_tournies = get_all_tournies(AUTH_TOKEN, query, SF_BASED_COORDS, SF_RADIUS, NUM_PER_PAGE)
sac_tournies = get_all_tournies(AUTH_TOKEN, query, SAC_BASED_COORDS, SAC_RADIUS, NUM_PER_PAGE)
tournies = sac_tournies + bay_tournies
np_tournies = pd.DataFrame(tournies).explode('events')
flat_tournies = flatten_nested_json_df(np_tournies)

In [ ]:
# clean the spreadsheet to filter by game and attendance
# one always works, comment out one and try the other

# ult_tournies = flat_tournies[
#     flat_tournies['events'].apply(
#         lambda x: isinstance(x, dict) and x.get('videogame', {}).get('name') == 'Super Smash Bros. Ultimate'
#     ) &
#     flat_tournies['events'].apply(
#         lambda x: isinstance(x, dict) and x.get('numEntrants', 0) >= 16
#     )
# ]

# ult_tournies = ult_tournies.reset_index(drop=True)

##########

ult_tournies = flat_tournies[
    (flat_tournies['events.videogame.name'] == 'Super Smash Bros. Ultimate') &
    (flat_tournies['events.numEntrants'] >= 16)
]

ult_tournies = ult_tournies.reset_index(drop=True)
ult_tournies

In [ ]:
# parse the spreadsheet to generate relevant columns + display
# one always works, comment out one and try the other

# ult_tournies['startgg_url'] = ult_tournies['events'].apply(lambda x: x['slug'] if isinstance(x, dict) and 'slug' in x else None)
# ult_tournies['startgg_url'] = 'start.gg/' + ult_tournies['startgg_url'].astype('str')
# ult_tournies['Event Date'] = ult_tournies['startAt'].map(
#     lambda x: datetime.fromtimestamp(x).strftime('%Y-%m-%d') if pd.notnull(x) and isinstance(x, (int, float)) else None
# )
# ult_tournies['StartGG TOURNAMENT_ID'] = ult_tournies['startgg_url'].map(lambda url: url.split("/")[-3:-2][0])
# ult_tournies['StartGG EVENT_ID'] = ult_tournies['startgg_url'].map(lambda url: url.split("/")[-1:][0]) 
# ult_tournies = ult_tournies.drop_duplicates('startgg_url', keep='first')

# ult_tournies['jakeSlug'] = ult_tournies['slug'] + '/event/' + ult_tournies['StartGG EVENT_ID']
# ult_tournies

####################

ult_tournies['startgg_url'] = 'start.gg/' + ult_tournies['events.slug'].astype(str)

# Convert startAt to a readable date format
ult_tournies['Event Date'] = ult_tournies['startAt'].map(
    lambda x: datetime.fromtimestamp(x).strftime('%Y-%m-%d') if pd.notnull(x) and isinstance(x, (int, float)) else None
)

# Extract StartGG Tournament ID and Event ID from the formatted slug
ult_tournies['StartGG TOURNAMENT_ID'] = ult_tournies['startgg_url'].map(lambda url: url.split("/")[-3] if len(url.split("/")) >= 3 else None)
ult_tournies['StartGG EVENT_ID'] = ult_tournies['startgg_url'].map(lambda url: url.split("/")[-1] if len(url.split("/")) >= 1 else None)

# Remove duplicates based on the tournament URL
ult_tournies = ult_tournies.drop_duplicates('startgg_url', keep='first')

# Create the jakeSlug column for Braacket usage
ult_tournies['jakeSlug'] = ult_tournies['slug'] + '/event/' + ult_tournies['StartGG EVENT_ID']
ult_tournies

In [ ]:
# go through df to remove unwanted brackets
def filter_dataframe(df):
    indices_to_keep = []
    
    for index, row in df.iterrows():
        while True:
            response = input(f"\nIndex: {index}\nName: {row['name']}\nStartGG URL: {row['startgg_url']}\nKeep this entry? (Y/N, default is Y): ").strip().upper()
            
            if response in ["", "Y"]:
                indices_to_keep.append(index)
                break
            elif response == "N":
                break
            else:
                print("Invalid input. Please enter 'Y' to keep or 'N' to remove.")
    
    return df.loc[indices_to_keep].reset_index(drop=True)

In [ ]:
# action of filtering the df
df_new = filter_dataframe(ult_tournies)
df_new.head()

### 1.3: Retrieving sets per tourney

Using Jake's code, we will retrieve all of the sets per event. This will also help us collect the attendance per player by parsing through the lists. 

In [ ]:
# retrieve numerical event IDs for each event
jake_slugs = df_new['jakeSlug'].astype(str).tolist()
eventIDs = [getEventID(tourney) for tourney in jake_slugs]
df_new['eventID'] = eventIDs
df_new.head()

In [ ]:
# keep relevant columns of df
df_clean = df_new[['name', 'city', 'startgg_url', 'Event Date', 'jakeSlug', 'eventID']].copy()
df_clean.head()

In [ ]:
# get numerical set IDs from event IDs
df_clean["setIDs"] = df_clean["eventID"].apply(lambda ID: getSetIDs(ID))
df_clean.head()

In [ ]:
# function to get actual sets from the set ids
cache = {}

def process_sets(set_id_list):
    sets = []
    for setID in set_id_list:
        if setID in cache:
            smashSet = cache[setID]
        else:
            smashSet = getPlayersAndScore(setID, cache)
            cache[setID] = smashSet
        sets.append(smashSet)
    return sets

In [ ]:
# applying the function to the df
df_clean["sets"] = df_clean["setIDs"].apply(process_sets)
df_clean.head()

In [ ]:
# checkpoint as this is where the majority of processing is over
df_clean.to_csv('chkpt1.csv')
df_clean = pd.read_csv('chkpt1.csv')

In [ ]:
# clean for tourney use
tourney_df = df_clean[['name', 'city', 'startgg_url', 'Event Date', 'sets']].copy()
tourney_df.head()

### 1.4: Preliminary Player Data Collection
Using the set data collected so far, we will generate a secondary dataframe with the following information:
- Player Tag
- Attendance
- Amount of sets played
- Wins
- Losses
- Head to Head Data
    - Positive
    - Even
    - Negative
- Loss to attendance ratio 

In [ ]:
# convert sets to list, get players
tourney_df["sets"] = tourney_df["sets"].apply(ast.literal_eval)
tourney_df["players"] = tourney_df["sets"].apply(playerList)
tourney_df.head()

In [ ]:
# retrieve attendance
def count_player_tournaments(players_lists):
    player_counts = Counter()
    
    for players in players_lists:
        if isinstance(players, set):
            player_counts.update(players)
    
    return sorted(player_counts.items(), key=lambda x: x[1], reverse=True)

player_tournament_counts = count_player_tournaments(tourney_df["players"].tolist())
player_tournament_counts[:5]

In [ ]:
# get wins losses h2h data and set count
def create_player_summary(sets_column):
    """
    Processes the sets column from the DataFrame and computes wins, losses, total sets, and head-to-head records.
    
    Args:
        sets_column (list): A list of dictionaries representing sets played.
    
    Returns:
        pd.DataFrame: A DataFrame with player stats including wins, losses, total sets, and head-to-head records.
    """
    player_stats = defaultdict(lambda: {
        "wins": 0, 
        "losses": 0, 
        "h2h": defaultdict(lambda: [0, 0]),
        "won_against": [],
        "lost_against": []
    })

    # Process each tournament's set data
    for sets in sets_column:
        for match in sets:
            players = list(match.keys())
            if len(players) < 2:
                continue  # Skip invalid matches
            
            p1, p2 = players[0], players[1]
            p1_score, p2_score = match[p1], match[p2]

            # **Skip matches where any score is None**
            if p1_score is None or p2_score is None:
                print(f"Skipping match due to missing score: {match}")
                continue  

            if p1_score > p2_score:
                player_stats[p1]["wins"] += 1
                player_stats[p2]["losses"] += 1
                player_stats[p1]["h2h"][p2][0] += 1  # p1 won
                player_stats[p2]["h2h"][p1][1] += 1  # p2 lost
                player_stats[p1]["won_against"].append(p2)
                player_stats[p2]["lost_against"].append(p1)
            elif p2_score > p1_score:
                player_stats[p2]["wins"] += 1
                player_stats[p1]["losses"] += 1
                player_stats[p2]["h2h"][p1][0] += 1  # p2 won
                player_stats[p1]["h2h"][p2][1] += 1  # p1 lost
                player_stats[p2]["won_against"].append(p1)
                player_stats[p1]["lost_against"].append(p2)

    # Convert player stats to DataFrame
    player_summary = []
    for player, stats in player_stats.items():
        total_sets = stats["wins"] + stats["losses"]
        pos_h2h = []
        even_h2h = []
        neg_h2h = []

        for opponent, (wins, losses) in stats["h2h"].items():
            record = f"{wins}-{losses}"
            if wins > losses:
                pos_h2h.append((opponent, record))
            elif wins == losses and wins > 0:  # Avoid 0-0 cases
                even_h2h.append((opponent, record))
            elif wins < losses:
                neg_h2h.append((opponent, record))

        player_summary.append({
            "Player": player,
            "Wins": stats["wins"],
            "Losses": stats["losses"],
            "Total Sets": total_sets,
            "Positive H2H": pos_h2h,
            "Even H2H": even_h2h,
            "Negative H2H": neg_h2h,
            "Won Against": stats["won_against"],
            "Lost Against": stats["lost_against"]
        })

    return pd.DataFrame(player_summary).sort_values(by="Total Sets", ascending=False).reset_index(drop=True)

# Apply the function to process the sets column
df_player_summary = create_player_summary(tourney_df["sets"])
df_player_summary.head()

In [ ]:
# attendance and loss to tourney ratio
player_tourney_dict = dict(player_tournament_counts)
df_player_summary["Tournaments Attended"] = df_player_summary["Player"].map(player_tourney_dict).fillna(0).astype(int)

# Calculate loss-to-tournament ratio
df_player_summary["Loss to Tournament Ratio"] = df_player_summary["Losses"] / df_player_summary["Tournaments Attended"]

df_player_summary.head()

### 1.5: ELO Generation 
Merge all of the player and set data into one, and then compute ELO. Afterwards we will populate the player data df with the following info:
- Assign players ELO
- Update wins to include ELO
- Update losses to include ELO
- Filter out notable wins and losses
- Add notable win to tournament ratio
- Add notable loss to tournament ratio 

In [ ]:
# retrieve all sets and players 
allsets = [match for sets in tourney_df["sets"] for match in sets]
allplayers = set().union(*tourney_df["players"])

playerMatrixIndex, gameMatrix, setMatrix = makeMatrices(allplayers, allsets)

In [ ]:
# generate elo 
elo = {player:1500 for player in allplayers}
print(elo)
for smashSet in allsets:
  elo = updateElo(elo, smashSet)
elo = sortElo(elo)

In [ ]:
# elo mapping
df_player_summary["ELO"] = df_player_summary["Player"].map(elo).fillna(0)
df_player_summary = df_player_summary.sort_values(by="ELO", ascending=False).reset_index(drop=True)
df_player_summary.head()

In [ ]:
# add notable info
def add_notable_results(df_summary, elo, players):
    bottom = len(players) - 30
    # Get top 30 players by ELO
    top_players = sorted(elo.keys(), key=lambda x: elo[x], reverse=True)[:30]
    # Get bottom 100 players by ELO
    bottom_players = sorted(elo.keys(), key=lambda x: elo[x])[:bottom]

    # Define a helper function to filter notable wins
    def filter_notable_wins(wins):
        return [player for player, _ in wins if player in top_players]

    # Define a helper function to filter notable losses
    def filter_notable_losses(losses):
        return [player for player, _ in losses if player in bottom_players]

    # Create the Notable Wins and Notable Losses columns
    df_summary["Notable Winning H2H"] = df_summary["Positive H2H"].apply(filter_notable_wins)
    df_summary["Notable Losing H2H"] = df_summary["Negative H2H"].apply(filter_notable_losses)

    return df_summary

In [ ]:
player_df = add_notable_results(df_player_summary, elo, allplayers)
player_df.to_csv('323data.csv')
player_df.head()
# export data as this is an important checkpoint